# SpellBreaker on IBM LinuxONE

This notebook records a live scoring run against the service on the LinuxONE Community Cloud VM. The transfer below is synthetic demo data; no real customer information is used.

In [ ]:
import importlib.metadata
import json
import platform
import time
from urllib.request import Request, urlopen

BASE_URL = 'http://127.0.0.1:8000'
def get_json(path):
    with urlopen(BASE_URL + path, timeout=10) as response:
        return json.loads(response.read())

health = get_json('/health')
report = get_json('/report')
assert platform.machine() == 's390x', platform.machine()
assert health['arch'] == 's390x' and health['trained_on'] == 's390x', health
assert report['trained_on'] == 's390x', report.get('trained_on')
packages = {name: importlib.metadata.version(name) for name in ('numpy', 'pandas', 'scikit-learn', 'scipy', 'joblib')}
print(json.dumps({'python': platform.python_version(), 'machine': platform.machine(), 'health': health, 'packages': packages}, indent=2))

In [ ]:
reset_req = Request(BASE_URL + '/demo/reset', data=b'{}', headers={'Content-Type': 'application/json'}, method='POST')
with urlopen(reset_req, timeout=10) as response:
    print('demo_reset:', response.status)
payload = {
    'customer_id': 'C-TAN',
    'payee_account': 'MB-778-201-9931',
    'payee_name': 'Secure Govt Account',
    'amount': 20000,
    'source_pot': 'savings',
    'flags': {'on_call': True, 'pasted_account': True, 'rushed_score': 0.8}
}
score_req = Request(BASE_URL + '/score', data=json.dumps(payload).encode(), headers={'Content-Type': 'application/json'}, method='POST')
with urlopen(score_req, timeout=10) as response:
    decision = json.loads(response.read())
assert decision['level'] == 'CRITICAL', decision
print(json.dumps({key: decision[key] for key in ('decision_id', 'risk', 'level', 'scam_type', 'action', 'reasons', 'latency_ms', 'scored_on')}, indent=2))

In [ ]:
case = None
for _ in range(40):
    case = get_json('/case/' + decision['decision_id'])
    if case.get('status') == 'ready':
        break
    time.sleep(0.25)
assert case and case.get('status') == 'ready', case
print(json.dumps({'case_runtime': case.get('runtime'), 'case_status': case.get('status'), 'case_note': case.get('note')}, indent=2))

## Training snapshot

The model report served by the LinuxONE service includes temporal test metrics, age-band fairness, per-scam-type recall, ablations, and leave-one-type-out recall. Scores describe the synthetic simulator and are not a real-world effectiveness estimate.

In [ ]:
print(json.dumps({
    'trained_on': report['trained_on'],
    'difficulty': report.get('difficulty'),
    'generation_parameters': report.get('generation_parameters'),
    'full_metrics': report['full'],
    'fairness_by_age_band': report['fairness_by_age_band'],
}, indent=2))